In [1]:
# Step 1-1. パラメータセット（基本設定）

# parameters
adjust_rate = 1.0
pivot_month = "2026-01"

# from google.colab import drive
from pathlib import Path

# 1. Google Driveのマウント
# drive.mount('/content/drive')

# 2. 共通のベースディレクトリを定義
BASE_DIR = Path("C:/Users/h_ino/Code/DevLab/project/島根県/20260318セミナー/PoC-AI/県内旅館宿泊数")
# BASE_DIR = Path('C:\Users\h_ino\Code\DevLab\project\島根県\20260318セミナー\PoC-AI\県内旅館宿泊数\Datas')

# データが格納されているフォルダ（説明書の構造に合わせる）
DATA_DIR = BASE_DIR / "Datas"
OUTPUT_DIR = BASE_DIR / "outputs"
# DATA_DIR = "Datas"
# DATA_DIR = Path("Datas")

# 3. パラメータセット（基本設定）
PREDICTION_TARGET_YEAR = 2026
MAX_CAPACITY_PER_MONTH = 1450


# 4. 各入力ファイルのパス定義
# 島根県データ
SHIMANE_TOURISM_FILE = DATA_DIR / "Shimane/shimane_tourism.csv"
SHIMANE_STAY_FILE    = DATA_DIR / "Shimane/shimane_accommodation.csv"
SHIMANE_TRANS_FILE   = DATA_DIR / "Shimane/shimane_transport.csv"

# 出雲市データ
IZUMO_TOURISM_FILE   = DATA_DIR / "Izumo/izumo_tourism.csv"
IZUMO_STAY_FILE      = DATA_DIR / "Izumo/izumo_accommodation.csv"

# 大社町データ
TAISHA_STAY_FILE     = DATA_DIR / "Taisha/taisha_accommodation.csv"

# 旅館実績データ
RYOKAN_DATA_FILE     = DATA_DIR / "大社の旅館宿泊数/1.csv"
RYOKAN_DATA_FILE_2026     = DATA_DIR / "大社の旅館宿泊数/2.csv"

# 施設の月間上限受入人数
MAX_CAPACITY_PER_MONTH = 1450

# 設定内容を表示して確認
print(f"予測する年: {PREDICTION_TARGET_YEAR}年")
print(f"施設の月間上限受入人数: {MAX_CAPACITY_PER_MONTH}人")
print(f"島根県観光データ: {SHIMANE_TOURISM_FILE}、{SHIMANE_STAY_FILE}、{SHIMANE_TRANS_FILE}")
print(f"出雲市観光データ: {IZUMO_TOURISM_FILE}、{IZUMO_STAY_FILE}")
print(f"大社町観光データ: {TAISHA_STAY_FILE}")
print(f"旅館宿泊実績データ: {RYOKAN_DATA_FILE}")
print(f"旅館宿泊実績データ(2026): {RYOKAN_DATA_FILE_2026}")

予測する年: 2026年
施設の月間上限受入人数: 1450人
島根県観光データ: C:\Users\h_ino\Code\DevLab\project\島根県\20260318セミナー\PoC-AI\県内旅館宿泊数\Datas\Shimane\shimane_tourism.csv、C:\Users\h_ino\Code\DevLab\project\島根県\20260318セミナー\PoC-AI\県内旅館宿泊数\Datas\Shimane\shimane_accommodation.csv、C:\Users\h_ino\Code\DevLab\project\島根県\20260318セミナー\PoC-AI\県内旅館宿泊数\Datas\Shimane\shimane_transport.csv
出雲市観光データ: C:\Users\h_ino\Code\DevLab\project\島根県\20260318セミナー\PoC-AI\県内旅館宿泊数\Datas\Izumo\izumo_tourism.csv、C:\Users\h_ino\Code\DevLab\project\島根県\20260318セミナー\PoC-AI\県内旅館宿泊数\Datas\Izumo\izumo_accommodation.csv
大社町観光データ: C:\Users\h_ino\Code\DevLab\project\島根県\20260318セミナー\PoC-AI\県内旅館宿泊数\Datas\Taisha\taisha_accommodation.csv
旅館宿泊実績データ: C:\Users\h_ino\Code\DevLab\project\島根県\20260318セミナー\PoC-AI\県内旅館宿泊数\Datas\大社の旅館宿泊数\1.csv
旅館宿泊実績データ(2026): C:\Users\h_ino\Code\DevLab\project\島根県\20260318セミナー\PoC-AI\県内旅館宿泊数\Datas\大社の旅館宿泊数\2.csv


In [2]:
# Parameters
adjust_rate = 1.02
pivot_month = "2026-02-01"


In [3]:
# Step 1-2. setup（分析の準備）

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

# 分析に直接関係ない警告（注意書き）を非表示にして見やすく
warnings.filterwarnings('ignore')

# グラフで日本語を表示するための設定 (Google Colab / ローカル両対応)
try:
    import japanize_matplotlib
except ImportError:
    # 未インストールの場合のみインストールを実行
    if 'google.colab' in sys.modules:
        !pip install -q japanize-matplotlib
        import japanize_matplotlib
    else:
        pass

# グラフの見た目を整える（フォントや背景色）
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('ggplot')
# SNS(Seaborn)のスタイル設定。日本語フォントが優先されるようにする
sns.set(style="whitegrid", font="IPAexGothic")

# 読みやすい年月（例：2024-01）を作成する関数
def make_year_month_string(year, month):
    return f"{int(year):04d}-{int(month):02d}"

# 乱数（偶然の数値）を固定して、何度実行しても同じ結果が出るように
np.random.seed(42)

print("分析の準備が完了しました。")

分析の準備が完了しました。


In [4]:
# Step 6. 2026年実績を基に更新（CSV更新に特化）

from datetime import datetime

if pivot_month is None:
    print("pivot_month が None のためスキップ")
else:

    print("adjust_rate:", adjust_rate)
    print("pivot_month:", pivot_month)
# パス
    forecast_path = OUTPUT_DIR / "forecast_2026.csv"

# 読み込み
    df = pd.read_csv(forecast_path)
    df = df.sort_values("month")


# 日付変換
    df["month"] = pd.to_datetime(df["month"])

# 前提：df["month"] は datetime に変換済み
    pivot = pd.to_datetime(pivot_month)

# デバッグ用
    print("pivot:", pivot)
    print(df[["month", "last_adjusted_at"]])


# カラム初期化
    if "last_adjusted_at" not in df.columns:
        df["last_adjusted_at"] = ""

# 既に処理済みかチェック（冪等性）
    # already_done = df.loc[df["month"] == pivot, "last_adjusted_at"].astype(str).str.len().gt(0).any()
    target = df[df["month"] == pivot]

    already_done = (
        target["last_adjusted_at"].notna() &
        (target["last_adjusted_at"].astype(str).str.strip() != "")
    ).any()
    if already_done:
        print("既に処理済みのためスキップ")
    else:
    # 補正（あなたの減衰ロジックそのまま）
        mask = df["month"] > pivot
        for i, idx in enumerate(df.index[mask]):
            decay_rate = 1 + (adjust_rate - 1) * (0.5 ** i)
            df.loc[idx, "predicted_stays"] *= decay_rate

    # pivotに刻印
        from datetime import datetime
        now = datetime.now().strftime("%Y-%m-%d %H:%M")
        df.loc[df["month"] == pivot, "last_adjusted_at"] = now

        print(f"更新実行 pivot={pivot_month}, at={now}")

# 保存
    df.to_csv(forecast_path, index=False)

    print(df.head())

adjust_rate: 1.02
pivot_month: 2026-02-01
pivot: 2026-02-01 00:00:00
        month  last_adjusted_at
0  2026-01-01  2026-05-13 06:53
1  2026-02-01  2026-05-13 07:28
2  2026-03-01               NaN
3  2026-04-01               NaN
4  2026-05-01               NaN
5  2026-06-01               NaN
6  2026-07-01               NaN
7  2026-08-01               NaN
8  2026-09-01               NaN
9  2026-10-01               NaN
10 2026-11-01               NaN
11 2026-12-01               NaN
既に処理済みのためスキップ
       month  month_only  predicted_stays  last_adjusted_at
0 2026-01-01           1       851.666667  2026-05-13 06:53
1 2026-02-01           2       776.900000  2026-05-13 07:28
2 2026-03-01           3      1307.838900               NaN
3 2026-04-01           4      1210.954650               NaN
4 2026-05-01           5      1195.917337               NaN
